In [15]:
''' getting required libraries 
    nx : to build graph,
    node2vec: to convert graph to embeddings
    pandas,numpy for datapipeling and faster computation
'''
import networkx as nx
from node2vec import Node2Vec
import pandas as pd
import numpy as np

In [4]:
'''making a DF pipeline to our cleaned data set
   known as 'delhivery_cleaned_1.csv' 
   and checking its columns
'''
df=pd.read_csv('delhivery_cleaned_1.csv')
df.columns

Index(['data', 'trip_creation_time', 'route_schedule_uuid', 'route_type',
       'trip_uuid', 'source_center', 'source_name', 'destination_center',
       'destination_name', 'od_start_time', 'od_end_time',
       'start_scan_to_end_scan', 'is_cutoff', 'cutoff_factor',
       'cutoff_timestamp', 'actual_distance_to_destination', 'actual_time',
       'osrm_time', 'osrm_distance', 'factor', 'segment_actual_time',
       'segment_osrm_time', 'segment_osrm_distance', 'segment_factor',
       'log_segment_factor', 'log_segment_osrm_distance',
       'log_segment_osrm_time', 'log_segment_actual_time', 'log_factor',
       'chokepoint_flag', 'segment_name', 'log_actual_time',
       'log_start_scan_to_end_scan', 'log_cutoff_factor',
       'log_actual_distance_to_destination'],
      dtype='object')

In [5]:
'''getting a set of  unique destination center
   getting a set of  unique source center 
   we first applied unique() to get unique list of dst_center then applied set()
   to get a set of destination_hubs,source_hubs
   then since source_hubs,destination_hubs are set 
   we get the union of both of them
   to get a set of unique nodes
'''
destination_center = set(df['destination_center'].unique())
source_center = set(df['source_center'].unique())
nodes = source_center.union(destination_center)

In [6]:
'''
    since our data is already divided into 'training' and 'test'  we creating
    train_df and test_df base on the logic that column df['data'] == 'training'
    data frame column == training for train_df and test_df=='test'
    
'''
test_df = df[df['data'] =='test'].copy()
train_df = df[df['data']=='training'].copy()

In [7]:
''' since we have to capture weights of the 
    edges using the median actual-vs-OSRM we
    first groupby our data using source_name,destination name 
    then we applied aggregation function of median to both
    ratio of segment_factor and median of base_distance
    we reset index to 0,1,2,3,4.... for our data in row wise
'''
network_edge = df.groupby(['source_name','destination_name']).agg(
    median_ratio = ('segment_factor','median'),
    base_distance = ('segment_osrm_distance','median')
).reset_index()

In [8]:
'''
    we then created a directed graph 
   and then we use iteration to fill our Graph edges 
   with the weight using network_edge data frame and G.add_edge()
   which we have created earlier we gave it information 
   source_name and destination_name 
   then we printed total number of edges and number of nodes

'''
G = nx.DiGraph()
for index,row in network_edge.iterrows():
    G.add_edge(
        row['source_name'],
        row['destination_name'],
        weight = row['median_ratio'],
        distance = row['base_distance']
    )
print('Graph Created Successfuly')
print(f'Number of nodes {G.number_of_nodes()}')
print(f'Number of edges {G.number_of_edges()}')

Graph Created Successfuly
Number of nodes 1475
Number of edges 1491


In [9]:
''' 
    in node2vec we then creating our randoms walks of dimension 32 and each walk length is 15
    we have taken dimension=32 because it lies in goldie-lock zone since if we try to do 64 it makes 
    computation expensively and no significant changes in the Model Performance and we have taken 
    walk_length=15 so that each walker can go upto 15 nodes and by
    num_walks=4 we mean to say each node is performing 4 walks. and p=1.0 is for DFS and q=0.5 BFS
'''
node2vec = Node2Vec(G,dimensions=32,walk_length=15,num_walks=4,p=1.0,q=0.5)
print('Generating Structural graph embeddings for hubs')
'''
this is where our node2vec random walk is converted in to embeddings with window =10 ,min
window=10: This sets the context window, meaning the model looks at up to 10 nodes ahead and 10 nodes behind the 
target node in a random walk sequence to learn their structural relationships and network neighborhoods.

min_count=1: This ensures that every single node is included in the embedding space, 
preventing the model from discarding rare or low-degree nodes that only appear once in the random walks.

batch_words=4: This acts as the batch size for the underlying training loop, pushing the 
text/walk data to the operating system's threads 4 words (nodes) at a time to manage memory 
cache efficiency during gradient updates . and finally we get them saved in dictionary hub_embeddings.
'''
graph_model = node2vec.fit(window=10,min_count=1,batch_words=4)
hub_embeddings = {node:graph_model.wv[node] for node in G.nodes()}

Computing transition probabilities:   0%|          | 0/1475 [00:00<?, ?it/s]

Generating walks (CPU: 1): 100%|██████████| 4/4 [00:00<00:00, 19.71it/s]


Generating Structural graph embeddings for hubs


In [10]:
#printing our hub embeddings.
hub_embeddings

{'AMD_Memnagar (Gujarat)': array([-1.3258482 , -0.28990248, -0.0538108 , -0.18664034,  1.3794365 ,
        -0.9453869 ,  0.43991855,  0.5238337 , -0.05162987, -1.290623  ,
         1.0772848 , -0.8479585 ,  0.393857  ,  0.16244906, -0.54839754,
        -0.58862495, -1.1307346 , -0.47588134,  1.1997375 ,  0.9322781 ,
        -0.25621688,  0.9273896 ,  0.10182967,  0.17163412, -1.4499373 ,
        -0.02915537, -0.2376    , -0.70627713,  0.2249222 , -1.16005   ,
         0.00494806, -1.0112426 ], dtype=float32),
 'AMD_Satellite (Gujarat)': array([-0.50424206, -0.48998612, -0.02373377,  0.77944064, -0.6072988 ,
        -1.0895734 , -0.2561484 ,  0.74469656,  0.48237935, -0.31893325,
         0.8733719 , -0.00704019,  0.44536752, -0.8307699 , -0.3319958 ,
         0.9030271 ,  0.30972153,  0.78841513,  0.40108934,  1.3547689 ,
        -0.5272139 ,  0.8244084 , -0.8020313 ,  0.07582571, -0.5817545 ,
        -0.05072544, -1.2242615 , -1.1541444 ,  1.4538059 , -0.8294486 ,
        -0.91770667,

In [11]:
embedding_dim = 32
fallback_vector = np.zeros(embedding_dim)
def get_hub_features(df,embedding_dict):
    ''' 
    this function is basically to add the hub embeddings for 
    dst_hub embeddings src_hub_embeddings to our 
    orignaldataframe.
    we do the iteration in our rows of dataframe then
    in the src = we get the src_name
    in the dst = we get the dst_name
    in the src_vec we get the value of embeddings similarly for dest_vec 
    and then we finally add then store src_df,dst_df and then finally 
    we return the concatenetted dataframe which is horizontaaly saved.
    '''
    source_vector = []
    dest_vector = []
    for _,row in df.iterrows():
        src = row['source_name']
        dst = row['destination_name']
        src_vec = embedding_dict.get(src)
        dst_vec = embedding_dict.get(dst)
        source_vector.append(src_vec)
        dest_vector.append(dst_vec)
    src_df = pd.DataFrame(source_vector,columns=[f'src_emb_{i}' for i in range(embedding_dim)],index=df.index)
    dst_df = pd.DataFrame(dest_vector,columns=[f'dst_emb_{i}' for i in range(embedding_dim)],index=df.index)
    print(src_df)
    return pd.concat([df,src_df,dst_df],axis=1)
print('Mapping graph features back to tabular data splits')
''' we then get a new datframe which has 
    train_df columns and its hub embedding 
    similarly for test_enhance ''' 
train_enhanced = get_hub_features(train_df,hub_embeddings)
test_enhanced = get_hub_features(test_df,hub_embeddings)

Mapping graph features back to tabular data splits
        src_emb_0  src_emb_1  src_emb_2  src_emb_3  src_emb_4  src_emb_5  \
0       -1.665259  -0.338677   0.945135   0.450759   0.055340   0.443410   
1       -1.665259  -0.338677   0.945135   0.450759   0.055340   0.443410   
2       -1.665259  -0.338677   0.945135   0.450759   0.055340   0.443410   
3       -1.665259  -0.338677   0.945135   0.450759   0.055340   0.443410   
4       -1.665259  -0.338677   0.945135   0.450759   0.055340   0.443410   
...           ...        ...        ...        ...        ...        ...   
142497  -0.169545  -0.044595   0.210221  -0.040087   0.180336  -0.039404   
142498  -0.169545  -0.044595   0.210221  -0.040087   0.180336  -0.039404   
142499  -0.169545  -0.044595   0.210221  -0.040087   0.180336  -0.039404   
142500  -0.169545  -0.044595   0.210221  -0.040087   0.180336  -0.039404   
142501  -0.169545  -0.044595   0.210221  -0.040087   0.180336  -0.039404   

        src_emb_6  src_emb_7  src_em

In [12]:
''' we imported XGBRegressor 
    the selected the base features which has
    significant correlation with our target column 'segment_actual_time'
'''
from xgboost import XGBRegressor
base_features = [
    'cutoff_factor',
    'actual_distance_to_destination',
    'osrm_time',
    'osrm_distance',
    'segment_osrm_distance',
    'segment_osrm_time',
]
''' 
    then we are getting the column which are for embeddings using if in their column 'src_emb_','dst_emb_'
    then we saved the ultimate feature which contain base_features + embedding_feature
    then we splitted them on the basis of train_enhanced and test_enhanced.
'''
embedding_feature = [col for col in train_enhanced.columns if 'src_emb_' in col or 'dst_emb_' in col]
ultimate_features = base_features + embedding_feature
target_col = 'segment_actual_time'
X_train = train_enhanced[ultimate_features]
y_train = train_enhanced[target_col]
X_test = test_enhanced[ultimate_features]
y_test = test_enhanced[target_col]
'''
    now we are building our XGBRegressor 
    with number of estimator=2000,
    we have taken them 2000 because after that it is converging
    and maxiumum depth to be 10 which mean that it is spliited 2^10=1024
    and then we fitted our X_train to the model
'''
xgb_enhanced = XGBRegressor(
    n_estimators=2000,
    learning_rate=0.05,
    max_depth=10,
    subsample=0.8,
    colsample_bytree=0.7,
    random_state=42,
    n_jobs=-1
)

print(f"Training Graph-Enhanced XGBoost on {X_train.shape[1]} total features...")
xgb_enhanced.fit(X_train, y_train)

Training Graph-Enhanced XGBoost on 70 total features...


,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.7
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes 

In [19]:

print(type(X_test),type(y_test))

<class 'pandas.core.frame.DataFrame'> <class 'pandas.core.series.Series'>


In [11]:
from sklearn.metrics import mean_absolute_error, r2_score
''' we thus then predicted our ETA using first by fitting our 
    training data then by predicting
    xgb_enhanced.predict(X_test) 
    to actual_time/orignal time then predicted
'''
y_pred = xgb_enhanced.predict(X_test)

eval_df = pd.DataFrame({
    'trip_id': test_enhanced['trip_uuid'],
    'actual_time': y_test,
    'predicted_eta': y_pred
})

trip_df = eval_df.groupby('trip_id')[['actual_time', 'predicted_eta']].sum().reset_index()

trip_df['lower_bound'] = trip_df['actual_time'] * 0.85
trip_df['upper_bound'] = trip_df['actual_time'] * 1.15
trip_df['within_15_percent'] = trip_df['predicted_eta'].between(trip_df['lower_bound'], trip_df['upper_bound'])
''' we then check our goodness of fit using various metrices like
    mean_absolute_error,mean_squared_error,root_mean_squared_error,
    r2_score and printing them.
'''
print("\n--- Graph-Enhanced XGBoost Trip-Level Performance ---")
print(f"R² Segment Score: {r2_score(y_test, y_pred):.4f}")
print(f"Segment MAE: {mean_absolute_error(y_test, y_pred):.2f} minutes")
print(f"Percentage of trips within ±15% window: {trip_df['within_15_percent'].mean() * 100:.2f}%")


--- Graph-Enhanced XGBoost Trip-Level Performance ---
R² Segment Score: 0.4936
Segment MAE: 12.51 minutes
Percentage of trips within ±15% window: 47.03%


In [ ]:
''' Trip level Peformance is that
    Total_unique trips that are being evaluted is 4163 
    and Percentage of trips where ETA(Expected Time of arrival) is within actual_time ±15%: 47.03%
    which means that out of 100 trips approximately we predicte 47 of them correctly '''

In [14]:
"""
Delhivery ETA Prediction Engine: Spatial-Temporal Graph Architecture

Dependencies and Justifications:
--------------------------------
1. NumPy (np) & Pandas (pd):
   - Used for structural data preprocessing, handling tabular shipment logs, 
     and executing vectorized matrix transformations on distance and time features.

2. PyTorch (torch, nn, F):
   - Provides the underlying deep learning framework. 
   - 'nn' is utilized to construct custom neural network layers and loss functions, 
     while 'F' (Functional) supplies activation functions (e.g., ReLU) to introduce 
     non-linearity into the network.

3. PyTorch Geometric (Data):
   - Crucial for modeling the logistics network as a graph structure.
   - 'Data' objects wrap node features (e.g., hub capacities, coordinates), edge indices 
     (e.g., transit routes, connectivity), and targets (e.g., actual transit times) 
     into geometric graph representations compatible with GNN layers.

4. Scikit-Learn Metrics (mean_absolute_error, r2_score):
   - Used to rigorously validate model performance on test splits.
   - Mean Absolute Error (MAE) quantifies the average expected deviation in minutes 
     for a given ETA, while the R² score evaluates the proportion of variance 
     explained by the spatial-temporal model.
"""
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from sklearn.metrics import mean_absolute_error, r2_score

print("Preparing graph data structures...")
'''  
    since our data is already divided into 'training' and 'test'  we creating
    train_df and test_df base on the logic that column df['data'] == 'training'
    data frame column == training for train_df and test_df=='test'
'''
train_df = df[df['data'] == 'training'].copy()
test_df = df[df['data'] == 'test'].copy()
"""
Graph Node Initialization & Hub Indexing
What this block does and why:
- unique_hubs: Grabs every single unique hub from both the source and destination columns. 
  This maps out the complete footprint of our logistics network so no hub gets left behind.
- hub_to_id: Converts messy string/categorical Hub IDs into clean, 0-indexed integers. 
  PyTorch Geometric needs integer indices to build the edge_index tensor ($2 \times E$).
- num_nodes: Tracks the total number of hubs ($V$). We pass this directly into our 
  embedding layers so the model knows how big the network structure is.
"""
unique_hubs = pd.concat([train_df['source_center'], train_df['destination_center']]).unique()
hub_to_id = {hub: i for i, hub in enumerate(unique_hubs)}
num_nodes = len(unique_hubs)
#in selected columns we had only those columns which has high correlation
edge_features = [
    'cutoff_factor',
    'actual_distance_to_destination',
    'osrm_time',
    'osrm_distance',
    'segment_osrm_distance'
]

def create_pyg_graph(target_df, hub_mapping, total_nodes, feature_cols, is_training=True):
     """
    Converts Delhivery shipment logs into a PyTorch Geometric Graph
    
    Why we do it this way:
    - Filtering: We drop any rows where the source or destination isn't in our master 
      hub mapping to avoid out-of-bounds indexing errors during training.
    - edge_index: Maps out our route topology as a 2 x E tensor. It tells the GNN 
      exactly which hub connects to which hub for every active delivery segment.
    - edge_attr: Converts our tabular shipment features (like distance, traffic, or delays) 
      into edge attributes. This ensures the model factors in route-specific weights, 
      not just the graph structure.
    - y: Our target variable (actual transit time) reshaped to (E, 1) so PyTorch 
      can compute regression loss per delivery segment.
    - x: Generates an identity matrix as node features. Since we don't have static physical 
      hub features here, this acts as a one-hot encoding for each hub, giving the model a 
      unique learnable embedding vector per location.
    """
    valid_rows = target_df[target_df['source_center'].isin(hub_mapping) & target_df['destination_center'].isin(hub_mapping)]

    src_ids = valid_rows['source_center'].map(hub_mapping).values
    dst_ids = valid_rows['destination_center'].map(hub_mapping).values

    edge_index = torch.tensor(np.array([src_ids, dst_ids]), dtype=torch.long)

    edge_attr_mat = valid_rows[feature_cols].apply(pd.to_numeric, errors='coerce').fillna(0).values
    edge_attr = torch.tensor(edge_attr_mat, dtype=torch.float)

    y = torch.tensor(valid_rows['segment_actual_time'].values, dtype=torch.float).unsqueeze(1)

    x = torch.eye(total_nodes, dtype=torch.float)

    pyg_data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y)
    return pyg_data, valid_rows
''' using create_pyg() function to convert our data frame to train_graph
    similarly for test_graph in clean_train_df we have valid rows which is in both source center mapping and dest map '''
train_graph, clean_train_df = create_pyg_graph(train_df, hub_to_id, num_nodes, edge_features)
test_graph, clean_test_df = create_pyg_graph(test_df, hub_to_id, num_nodes, edge_features)

print(f"Train Graph: {train_graph}")
print(f"Test Graph: {test_graph}\n")

#we imported because we want to implemt sample aggregate graph and then its neural nework.
from torch_geometric.nn import SAGEConv

class DelhiveryGraphSAGE(nn.Module):
    """
    GraphSAGE Architecture for Delhivery ETA Prediction
    
    How this model works and why it's structured this way:
    - Two-layer SAGEConv: The first layer passes messages between directly connected hubs 
      to get local traffic/flow context. The second layer expands this to look two hops 
      out, capturing wider regional network congestion.
    - Node Embedding Generation (src & dst): Inside the forward pass, we extract the 
      final embeddings for the specific source hubs (row) and destination hubs (col) 
      making up our active shipping routes.
    - Feature Concatenation (combined): We glue together the source hub vector, the 
      destination hub vector, and the raw trip edge attributes (like distance or weather). 
      This creates a comprehensive snapshot of the entire delivery segment.
    - MLP Regression Decoder: A deep Multi-Layer Perceptron that takes this massive 
      combined feature vector, passes it through non-linear layers (ReLU), drops out 
      10% of neurons to prevent overfitting, and squashes everything down to a 
      single continuous scalar output—our final predicted ETA.
    """
    def __init__(self, num_nodes, node_dim, edge_dim, hidden_dim):
        super(DelhiveryGraphSAGE, self).__init__()

        self.sage1 = SAGEConv(num_nodes, hidden_dim)
        self.sage2 = SAGEConv(hidden_dim, node_dim)

        self.mlp = nn.Sequential(
            nn.Linear(node_dim + node_dim + edge_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, data):
        """
        Executes the network forward pass to map the graph to an ETA prediction.
        Step-by-step breakdown:
        1. Message Passing (h): We pass the one-hot node features through two layers of 
           GraphSAGE. Layer 1 captures immediate neighbor hub context (1-hop) and applies 
           a ReLU. Layer 2 pushes this out to 2-hops to capture regional network state.
        2. Unpacking Edges (row, col): Split the edge index into source nodes (row) and 
           destination nodes (col) for all active shipment segments in the current batch.
        3. Indexing Embeddings: Slice the master node embedding tensor 'h' to extract the 
           specific spatial vectors for our source hubs and destination hubs.
        4. Feature Fusion (combined): Glue the source embedding, destination embedding, 
           and physical trip features (edge_attr like distance/time matrix data) side-by-side.
        5. Prediction: Pass this combined vector through the MLP decoder to output the 
           final continuous scalar value (estimated travel time) for each segment.
        """
        h = F.relu(self.sage1(data.x, data.edge_index))
        h = self.sage2(h, data.edge_index)
        row, col = data.edge_index
        src_embeddings = h[row]
        dst_embeddings = h[col]
        combined = torch.cat([src_embeddings, dst_embeddings, data.edge_attr], dim=1)
        predictions = self.mlp(combined)
        return predictions
#this is for using nvidia gpu if available.
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
''' now we implementing our model with following features
    and printing our model
'''
model = DelhiveryGraphSAGE(
    num_nodes=num_nodes,
    node_dim=16,
    edge_dim=len(edge_features),
    hidden_dim=64
).to(device)

print("Model Layout Initialized:")
print(model)
'''
we then used Adaaptive moment generation to dynamically change learning rate to 0.01 and the 
criterion is MSELoss().
'''
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
criterion = nn.MSELoss()
#we then feed our train graph to device AND test GRAPH too
train_graph = train_graph.to(device)
test_graph = test_graph.to(device)

print("\nStarting GNN Training Loop...")
model.train()
for epoch in range(1, 406):
    """
    Standard PyTorch training loop running for 405 epochs.
    How each iteration updates the graph model:
    - zero_grad(): Wipes out the gradients from the previous step so they 
      don't accumulate on top of each other.
    - model(train_graph): Feeds the entire graph structure into our GNN 
      to get predictions for all delivery segments.
    - loss.backward(): Computes the error against actual actual travel times (y) 
      and propagates it backward through the network layers.
    - optimizer.step(): Adjusts the model's weights to minimize the loss.
    - print logging: Triggers a snapshot every 15 epochs (and the first one) 
      to track convergence without cluttering the console.
    """
    optimizer.zero_grad()

    predictions = model(train_graph)
    loss = criterion(predictions, train_graph.y)
    loss.backward()
    optimizer.step()

    if epoch % 15 == 0 or epoch == 1:
        print(f"Epoch {epoch:03d} / 405 | Training Loss (MSE): {loss.item():.4f}")

model.eval()
with torch.no_grad():
    ''' sending our prediction to 
        cpu -> to numpy -> from 2d to 1d'''
    test_predictions = model(test_graph).cpu().numpy().flatten()
    actual_times = test_graph.y.cpu().numpy().flatten()
'''getting the metrics of r2_score() and MAE()'''
r2 = r2_score(actual_times, test_predictions)
mae = mean_absolute_error(actual_times, test_predictions)
#printig our Model Metrics
print("\n==================================================")
print("--- GraphSAGE Evaluation Metrics ---")
print(f"Segment R² Score: {r2:.4f}")
print(f"Segment Mean Absolute Error (MAE): {mae:.2f} minutes")
''' we thus calculated our trip level performnace
    we first created a evaluting DataFrame 
    and then a trip df which contain groupby data based trip id basically to add
    the segment wise data and reset_index for row to be in 0,1,2,3,4....
'''
eval_df = pd.DataFrame({
    'trip_id': clean_test_df['trip_uuid'],
    'actual_time': actual_times,
    'predicted_eta': test_predictions
})
''' we calculating our percentage of trips that has achieved between
    15_percent and then we finally printing Percentage of trips where 
    ETA is within actual_time ±15%
'''
trip_df = eval_df.groupby('trip_id')[['actual_time', 'predicted_eta']].sum().reset_index()

trip_df['lower_bound'] = trip_df['actual_time'] * 0.85
trip_df['upper_bound'] = trip_df['actual_time'] * 1.15
trip_df['within_15_percent'] = trip_df['predicted_eta'].between(trip_df['lower_bound'], trip_df['upper_bound'])

percentage_accurate_trips = trip_df['within_15_percent'].mean() * 100

print(f"Total Evaluated Unique Trips: {len(trip_df)}")
print(f"Trip-Level Accuracy (Within ±15% Window): {percentage_accurate_trips:.2f}%")
print("==================================================")

Preparing graph data structures...
Train Graph: Data(x=[1590, 1590], edge_index=[2, 103126], edge_attr=[103126, 5], y=[103126, 1])
Test Graph: Data(x=[1590, 1590], edge_index=[2, 38765], edge_attr=[38765, 5], y=[38765, 1])

Model Layout Initialized:
DelhiveryGraphSAGE(
  (sage1): SAGEConv(1590, 64, aggr=mean)
  (sage2): SAGEConv(64, 16, aggr=mean)
  (mlp): Sequential(
    (0): Linear(in_features=37, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
    (3): Linear(in_features=64, out_features=32, bias=True)
    (4): ReLU()
    (5): Linear(in_features=32, out_features=1, bias=True)
  )
)

Starting GNN Training Loop...
Epoch 001 / 405 | Training Loss (MSE): 3977.9541
Epoch 015 / 405 | Training Loss (MSE): 2840.7644
Epoch 030 / 405 | Training Loss (MSE): 2200.0034
Epoch 045 / 405 | Training Loss (MSE): 2030.3256
Epoch 060 / 405 | Training Loss (MSE): 1848.7227
Epoch 075 / 405 | Training Loss (MSE): 1751.3684
Epoch 090 / 405 | Training Loss (MSE): 1691.2303

In [ ]:
''' Trip level Peformance is that
    Total_unique trips that are being evaluted is 4163 
    and Percentage of trips where ETA(Expected Time of arrival) is within actual_time ±15%: 40.44%
    which means that out of 100 trips approximately we predicte 40 of them correctly which is less than node2vec conversion '''